# Kalpa Health data pack: cleaning, one folder at a time

The raw data pack is a zip of messy files: CSVs in two encodings, JSON Lines with nested data, gzip files, timestamps in three formats, money in rupees **and** paise. This notebook turns it into **clean, joinable tables**.

Every section follows the same four steps: **read, clean, check, save.**

| Section | Folder | What we do |
|---|---|---|
| 0 | setup | mount Drive, unzip, find the data root |
| 1 | helpers | small reusable functions |
| 2 | `reference/` | read lookup tables, price to number |
| 3 | `patients/` | one row per patient |
| 4 | `bookings/legacy/` (CSV) | parse `dd/mm/YYYY` dates |
| 5 | `bookings/nova/` (JSON Lines) | flatten nested JSON, fix timezones |
| 6 | legacy + nova | stack, standardise, dedupe, join the test catalogue |
| 7 | `results/` | dedupe, find orphans, join to lines |
| 8 | booking-level table | collapse lines to one row per booking |
| 9 | `collections/` | dedupe, parse dates, find orphans |
| 10 | `invoices/` | numeric amount, date, find orphans |
| 11 | `payments/` | paise to rupees, epoch to datetime, normalise invoice ids, 3-way join |
| 12 | `campaigns/` | join tags to the campaign table |
| 13 | `plan/` | wide to long with `melt` |
| 14 | outputs | write clean files |
| 15 | sanity checks | cities, systems, payments |

**The pattern repeated in nearly every section is orphan checking.** An *orphan* is a row whose key has no parent (for example an invoice for a booking that does not exist). We never drop orphans silently. We count them and **save them to a separate file**, so someone can investigate.



## 0. Setup (Colab)

### Imports

- `glob`: find files with wildcards (`*.csv.gz`)
- `gzip`: open compressed files
- `json`: read JSON text
- `os`: folders and paths
- `gc`: free memory
- `zipfile`: unzip the data pack
- `pandas` as `pd`

**What we used:** `import a, b, c` on one line, `import ... as ...`.

In [43]:
import glob, gzip, json, os, gc, zipfile
import pandas as pd

In [44]:
# --- 1. mount Google Drive (a popup asks you to sign in the first time)
from google.colab import drive
drive.mount("/content/drive/")

Mounted at /content/drive/


### Paths

- `ZIP`: where the data pack lives on Drive. **Edit this** if your zip sits inside a folder
- `OUT`: where the clean files go. It is on **Drive**, so a runtime reset cannot wipe your results
- `os.makedirs(..., exist_ok=True)` creates the folder if missing and does nothing if it exists

**What we used:** constants in CAPITALS (a convention for "settings"), `os.makedirs`.

In [45]:
# --- 2. paths: edit ZIP if your zip is inside a Drive folder
ZIP = "/content/drive/MyDrive/Build1_DataPack.zip"
OUT = "/content/drive/MyDrive/merged"      # on Drive, so results survive a runtime reset
os.makedirs(OUT, exist_ok=True)

### Unzip to local disk and find the data root

Reading thousands of files from Drive is slow, so we unzip onto Colab's **local** disk.

- `CANDIDATES`: the folders where the data might land
- `find_root()` returns the first candidate that really contains `reference/tests_catalogue.csv` (a file we know must exist)
- if nothing is found, check the zip exists (`assert`), unzip, and look again
- a final `assert` stops with a clear message if we still cannot find it. Failing early with a clear message is better than a confusing error 200 lines later

**What we used:** a function with a loop and `return`, `os.path.exists`, `assert cond, "message"`, `with zipfile.ZipFile(...) as z`, `z.extractall`.

In [46]:
# --- 3. unzip to Colab's local disk (much faster to read than Drive) and locate the data root
CANDIDATES = ["/content/datapack", "/content/Build1_DataPack/datapack", "/content/Build1_DataPack"]

def find_root():
    for p in CANDIDATES:
        if os.path.exists(f"{p}/reference/tests_catalogue.csv"):
            return p

ROOT = find_root()
if ROOT is None:
    assert os.path.exists(ZIP), f"zip not found at {ZIP}: check the path or upload it to My Drive"
    print("unzipping ...")
    with zipfile.ZipFile(ZIP) as z:
        z.extractall("/content")
    ROOT = find_root()
assert ROOT, "could not find reference/tests_catalogue.csv: run  !find /content -maxdepth 3 -type d  and set ROOT by hand"

### Print what we found

A quick visual check: the root, the output folder, and the folders inside the pack.

**What we used:** `print` with several arguments, `sorted()` over a generator, `os.listdir`, `os.path.isdir`.

In [47]:
print("ROOT =", ROOT)
print("OUT  =", OUT)
print("folders:", sorted(d for d in os.listdir(ROOT) if os.path.isdir(f"{ROOT}/{d}")))

ROOT = /content/datapack
OUT  = /content/drive/MyDrive/merged
folders: ['bookings', 'campaigns', 'collections', 'invoices', 'patients', 'payments', 'plan', 'reference', 'results']


### Show free memory

Some cells load large tables. Knowing the free RAM tells you how much headroom there is. `try/except ImportError` means: if the `psutil` library is missing, skip silently.

**What we used:** `try / except`, `psutil.virtual_memory()`, dividing by `1e9` for GB.

In [48]:
# --- 4. show available RAM so you know how much headroom there is
try:
    import psutil
    print(f"RAM: {psutil.virtual_memory().available/1e9:.1f} GB free of {psutil.virtual_memory().total/1e9:.1f} GB")
except ImportError:
    pass

RAM: 10.5 GB free of 13.6 GB


## 1. Helper functions

**Why read everything as text (`dtype=str`)?** If pandas guesses types, IDs like `PHPUN035` or `000123` can turn into floats and lose their leading zeros, and one file where a column is entirely blank gets `float64` while another gets `object`. So we read **everything as text** and convert to numbers and dates **deliberately, one column at a time**.

### `clean_text(df)`

Two fixes applied to every column:
1. strip leading/trailing spaces and turn empty strings into real missing values (`NaN`)
2. drop rows that are a **repeat of the header** (happens when files were glued together)

How the header-echo check works: `df.eq(pd.Series(df.columns, ...), axis=1)` compares each cell with its own column name. If **all** cells in a row equal their column names, the row is the header repeated.

**What we used:** `for` loop over columns, `.astype("string")`, `.str.strip()`, `.mask(condition)`, `.eq(...)`, `.all(axis=1)`, `~` (not), `reset_index(drop=True)`.

In [49]:
def clean_text(df):
    # 1) trim whitespace in every column and turn empty strings into missing values
    for c in df.columns:
        s = df[c].astype("string").str.strip()
        df[c] = s.mask(s == "")
    # 2) drop rows that are a repeat of the header (happens when files were glued together)
    echo = df.eq(pd.Series(df.columns, index=df.columns), axis=1).fillna(False).all(axis=1)
    if echo.any():
        print(f"  dropped {echo.sum()} repeated header rows")
        df = df[~echo].reset_index(drop=True)
    return df

### `read_csv_any(path)`

We do not know the file encoding. Try UTF-8, then Windows-1252, then Latin-1 (which never fails, because it can decode any byte). The first one that works wins. Then the result goes through `clean_text`.

**What we used:** `for ... in (tuple)`, `try / except UnicodeDecodeError`, `break` and `continue`, `pd.read_csv(dtype=str, encoding=...)`.

In [50]:
def read_csv_any(path):
    # encoding is unknown, so try UTF-8, then Windows-1252, then latin-1 (which never fails)
    for enc in ("utf-8", "cp1252", "latin-1"):
        try:
            df = pd.read_csv(path, dtype=str, encoding=enc)
            break
        except UnicodeDecodeError:
            continue
    return clean_text(df)

### `read_many(pattern)`

Reads **every file matching a wildcard** and stacks them into one table. If `add_source=True`, it adds a `source_file` column, so any bad row can be traced back to its original file ("lineage").

**What we used:** `glob.glob(pattern, recursive=True)`, a list of DataFrames, `pd.concat(frames, ignore_index=True)`, `os.path.relpath`, default argument (`add_source=True`).

In [51]:
def read_many(pattern, add_source=True):
    # read every file matching a glob and stack them
    files = sorted(glob.glob(pattern, recursive=True))
    if not files:
        print(f"WARNING no files match {pattern}")
        return pd.DataFrame()
    frames = []
    for f in files:
        d = read_csv_any(f)
        if add_source:
            d["source_file"] = os.path.relpath(f)   # keep lineage so bad rows can be traced
        frames.append(d)
    out = pd.concat(frames, ignore_index=True)
    print(f"{pattern}: {len(files)} files, {len(out):,} rows")
    return out

### `to_num(s)`

Turns text like `"1,070.00"` into `1070.0`. Anything that cannot be a number becomes `NaN` (not an error).

**What we used:** `.str.replace(",", "", regex=False)`, `pd.to_numeric(errors="coerce")`.

In [52]:
def to_num(s):
    # "1,070.00" -> 1070.0 ; anything unparseable -> NaN
    return pd.to_numeric(s.astype("string").str.replace(",", "", regex=False), errors="coerce")

### `join_report(left, right, on, label)`

A left join that **prints how many left rows found no partner**. We use this instead of a plain `merge` everywhere, so a bad join announces itself.

How it works: `indicator=True` adds a column `_merge` saying `both` / `left_only` / `right_only`. We count the `left_only` rows and then drop the helper column.

**What we used:** `merge(how="left", indicator=True)`, `m["_merge"] == "left_only"`, `.drop(columns=...)`.

In [53]:
def join_report(left, right, on, label):
    # left join + print how many left rows found no partner
    m = left.merge(right, on=on, how="left", indicator=True)
    print(f"[{label}] {(m['_merge'] == 'left_only').sum():,} of {len(left):,} rows unmatched on {on}")
    return m.drop(columns="_merge")

### `norm_city(s)`

The same city is written differently (`bangalore`, `Bengaluru`). We use a **dictionary** to map known variants to one name, and fall back to title case (`PUNE` becomes `Pune`) for the rest.

**What we used:** dictionary (`CITY_MAP`), `.str.lower()`, `.map(dict)`, `.fillna(other_series)`, `.str.title()`.

In [54]:
CITY_MAP = {"bangalore": "Bengaluru", "bengaluru": "Bengaluru"}

def norm_city(s):
    key = s.str.strip().str.lower()
    return key.map(CITY_MAP).fillna(s.str.strip().str.title())

### `free(*names)`

Large tables eat memory. `free("a", "b")` deletes variables **by name** and asks Python to give the memory back. Calling it twice is safe (`pop(n, None)` does nothing if the name is gone).

**What we used:** `*args` (any number of arguments), docstring, `globals().pop(name, None)`, `gc.collect()`.

In [55]:
def free(*names):
    """Delete big intermediate frames by name (safe to call twice) and hand the memory back."""
    for n in names:
        globals().pop(n, None)
    gc.collect()

### Quick test of `clean_text` and `norm_city` on a toy table

Always test a cleaning function on a tiny example **where you know the right answer** before running it on millions of rows. The toy table has: a padded value (`" a2 "`), a blank (`""`), a repeated header row (`"id", "city"`), and city names in the wrong case (`bangalore`, `  PUNE`) for `norm_city` to fix.

**What we used:** building a DataFrame from a dictionary of lists.

In [56]:
toy = pd.DataFrame({"id": ["a1", " a2 ", "id", ""], "city": ["bangalore", "  PUNE", "city", "Jaipur"]})
toy = clean_text(toy)
toy["city"] = norm_city(toy["city"])
toy

  dropped 1 repeated header rows


,id,city
0,a1,Bengaluru
1,a2,Pune
2,<NA>,Jaipur


## 2. `reference/`: lookup tables

Small files, so little cleaning. The only conversion is `list_price_rs` to a number, so we can later compare it with the price actually charged.

- `cat` = test catalogue (code, name, list price)
- `clinics` = clinic details
- `cities` = city list

**What we used:** `read_csv_any`, `to_num`, `.dtypes` (check that the conversion worked), `.head()`.

In [57]:
cat     = read_csv_any(f"{ROOT}/reference/tests_catalogue.csv")
clinics = read_csv_any(f"{ROOT}/reference/clinics.csv")
cities  = read_csv_any(f"{ROOT}/reference/cities.csv")

cat["list_price_rs"] = to_num(cat["list_price_rs"])
print(cat.dtypes)
cat.head()

test_code        string[python]
test_name        string[python]
list_price_rs             Int64
is_package       string[python]
dtype: object


,test_code,test_name,list_price_rs,is_package
0,T001,Complete blood count,350,0
1,T002,Fasting glucose,120,0
2,T003,HbA1c,450,0
3,T004,Lipid profile,650,0
4,T005,Liver function,700,0


### A note on `cities`

The original script loads `cities` but never joins it. We just peek at it.

**What we used:** `len()`, `.columns[0]`.

In [58]:
# the original script loads cities but never joins it
print(len(cities), "city rows; first column is", cities.columns[0])
clinics.head()

6 city rows; first column is city_code


,clinic_id,city,clinic_name,opened_on,has_lab,slots_per_day
0,AHM-01,Ahmedabad,Kalpa Health Ahmedabad 1,2023-06-01,1,60
1,AHM-02,Ahmedabad,Kalpa Health Ahmedabad 2,2023-06-01,0,40
2,AHM-03,Ahmedabad,Kalpa Health Ahmedabad 3,2023-06-01,0,40
3,AHM-04,Ahmedabad,Kalpa Health Ahmedabad 4,2023-06-01,0,40
4,AHM-05,Ahmedabad,Kalpa Health Ahmedabad 5,2023-06-01,0,40


## 3. `patients/`

One patient may appear more than once, so we keep the **first row per `patient_id`**. We then:
- drop `patient_name` (not needed downstream) and `city` (the booking already has the city)
- **rename** the remaining columns with a `patient_` prefix, so they cannot collide with booking columns after the join

`errors="ignore"` in `drop` means: if a column is not there, do not crash.

**What we used:** `read_many`, `.duplicated().sum()` (count repeats), method chaining in brackets, `drop_duplicates`, `.drop(columns=...)`, `.rename(columns={...})`.

In [59]:
patients = read_many(f"{ROOT}/patients/*.csv.gz", add_source=False)

print("duplicate patient_ids:", patients.duplicated("patient_id").sum())
patients_slim = (
    patients.drop_duplicates("patient_id")
            .drop(columns=["patient_name", "city"], errors="ignore")
            .rename(columns={"phone": "patient_phone", "dob": "patient_dob",
                             "signup_date": "patient_signup_date"})
)
patients_slim.head()

/content/datapack/patients/*.csv.gz: 1 files, 309,217 rows
duplicate patient_ids: 0


,patient_id,patient_phone,patient_dob,patient_signup_date
0,PAHM0030088,+91 85778 98441,1988-02-02,2023-01-14
1,PAHM0043743,+91 84824 41827,1992-11-13,2025-10-02
2,PAHM0011366,+91 77327 81379,1944-07-05,2023-01-18
3,PAHM0009021,+91 76067 53849,1958-03-06,2022-08-06
4,PAHM0052680,+91 81849 12697,2005-11-01,2026-05-19


## 4. `bookings/legacy/`: CSV, one row per booking and test

Dates look like `dd/mm/YYYY HH:MM`. We pass an **explicit format** so `03/04/2026` is never read as March 4th (day-first vs month-first is the classic date bug).

- `errors="coerce"`: a bad value becomes `NaT` (missing date) instead of crashing
- `bad` = rows that **had a value but failed to parse**. A blank that stays blank is not a failure, so we count `parsed.isna() & original.notna()`
- `source_system = "legacy"` labels where each row came from

**What we used:** `pd.to_datetime(format="%d/%m/%Y %H:%M", errors="coerce")`, `&` between masks, a `for` loop over two columns, f-strings.

In [60]:
legacy = read_many(f"{ROOT}/bookings/legacy/*/*.csv.gz")

for c in ["created_at", "slot_at"]:
    parsed = pd.to_datetime(legacy[c], format="%d/%m/%Y %H:%M", errors="coerce")
    bad = parsed.isna() & legacy[c].notna()
    print(f"{c}: {bad.sum():,} unparseable -> NaT")
    legacy[c] = parsed

legacy["source_system"] = "legacy"
legacy.head()

  dropped 1 repeated header rows
  dropped 1 repeated header rows
  dropped 1 repeated header rows
  dropped 1 repeated header rows
  dropped 1 repeated header rows
  dropped 1 repeated header rows
  dropped 1 repeated header rows
  dropped 1 repeated header rows
  dropped 1 repeated header rows
  dropped 1 repeated header rows
  dropped 1 repeated header rows
  dropped 1 repeated header rows
  dropped 1 repeated header rows
  dropped 1 repeated header rows
  dropped 1 repeated header rows
  dropped 1 repeated header rows
  dropped 1 repeated header rows
  dropped 1 repeated header rows
  dropped 1 repeated header rows
  dropped 1 repeated header rows
  dropped 1 repeated header rows
/content/datapack/bookings/legacy/*/*.csv.gz: 102 files, 1,252,765 rows
created_at: 0 unparseable -> NaT
slot_at: 0 unparseable -> NaT


,booking_id,created_at,slot_at,city,clinic_id,channel,source,patient_id,patient_name,test_code,test_price,home_fee,source_file,source_system
0,BK-AHM-202501-000001,2025-01-08 03:16:00,2025-01-09 08:26:00,Ahmedabad,AHM-03,clinic,walkin,PAHM0028525,Saanvi Bose,T009,800,0,datapack/bookings/legacy/ahmedabad/bookings_20...,legacy
1,BK-AHM-202501-000001,2025-01-08 03:16:00,2025-01-09 08:26:00,Ahmedabad,AHM-03,clinic,walkin,PAHM0028525,Saanvi Bose,T004,650,0,datapack/bookings/legacy/ahmedabad/bookings_20...,legacy
2,BK-AHM-202501-000001,2025-01-08 03:16:00,2025-01-09 08:26:00,Ahmedabad,AHM-03,clinic,walkin,PAHM0028525,Saanvi Bose,T007,550,0,datapack/bookings/legacy/ahmedabad/bookings_20...,legacy
3,BK-AHM-202501-000002,2025-01-03 05:18:00,2025-01-06 13:22:00,Ahmedabad,AHM-04,clinic,app,PAHM0031384,Nisha Iyer,T004,650,0,datapack/bookings/legacy/ahmedabad/bookings_20...,legacy
4,BK-AHM-202501-000003,2025-01-27 10:46:00,2025-01-28 13:37:00,Ahmedabad,AHM-01,home,app,PAHM0027001,Sneha Menon,T003,450,150,datapack/bookings/legacy/ahmedabad/bookings_20...,legacy


## 5. `bookings/nova/`: JSON Lines with nested data

Each line of the file is one booking, shaped like this:

```
{"booking_id": "...", "patient": {"patient_id": "...", ...}, "tests": [{"code": "...", "price": ...}, ...]}
```

So `patient` is a **dict inside the row** and `tests` is a **list inside the row**. To get a flat table we must:

1. read each gzip file line by line with `json.loads`
2. give bookings with an **empty** `tests` list one empty test, otherwise they would vanish when exploded
3. explode `tests` into one row per test
4. flatten the `patient` dict into `patient.*` columns
5. rename to the legacy column names so the two systems can be stacked
6. convert timestamps (ISO with a UTC offset) to India time and drop the timezone, to match legacy (naive local time)

### Steps 1 and 2a: read the lines

**What we used:** nested `for` loops, `gzip.open(path, "rt")` (read as text), `with ... as fh`, `json.loads`, `list.append`, `os.path.relpath`.

In [61]:
rows = []
for f in sorted(glob.glob(f"{ROOT}/bookings/nova/*/*.jsonl.gz")):
    with gzip.open(f, "rt", encoding="utf-8") as fh:
        for line in fh:
            if line.strip():
                obj = json.loads(line)
                obj["source_file"] = os.path.relpath(f)
                rows.append(obj)
print(f"{len(rows):,} nova bookings")

26,543 nova bookings


### Step 2b: give empty bookings one empty test

`r.get("tests")` is empty or missing for those bookings, so we set `tests = [{}]` (a list with one empty dict). After the explode, that booking still has one row with blank test fields.

**What we used:** `dict.get`, `not` on an empty list, assignment into a dict.

In [62]:
for r in rows:
    if not r.get("tests"):
        r["tests"] = [{}]

### Steps 3 to 5: explode, flatten, rename

- `meta_cols` = every booking-level key (taken from the first record) except `tests`. These are copied onto each exploded row
- `pd.json_normalize(rows, record_path="tests", meta=meta_cols)` creates **one row per test** and carries the booking fields along
- the nested `patient` dict becomes columns `patient.patient_id`, ... with a second `json_normalize` + `.add_prefix`
- `NOVA_MAP` renames nova's names to the legacy names (`tests.code` to `test_code`, and so on)

**What we used:** list comprehension with a condition, `pd.json_normalize(record_path, meta, record_prefix)`, `.add_prefix`, `pd.concat(axis=1)`, `.rename(columns=dict)`, `.tolist()`.

In [63]:
meta_cols = [k for k in rows[0] if k != "tests"]   # every booking-level key, taken from the first record
nova = pd.json_normalize(rows, record_path="tests", record_prefix="tests.", meta=meta_cols, errors="ignore")

# flatten the nested patient dict
p = pd.json_normalize(nova["patient"]).add_prefix("patient.")
nova = pd.concat([nova.drop(columns="patient"), p], axis=1)

NOVA_MAP = {"patient.patient_id": "patient_id", "tests.code": "test_code", "tests.price": "test_price"}
nova = nova.rename(columns=NOVA_MAP)
nova.columns.tolist()

['test_code',
 'test_price',
 'booking_id',
 'created_at',
 'slot_at',
 'city',
 'clinic_id',
 'channel',
 'source',
 'home_fee',
 'rescheduled_from',
 'status_at_export',
 'source_file',
 'patient_id',
 'patient.is_new']

### Step 6: clean the text, then convert the dates

**Order matters.** `clean_text` turns every column into text, so it must run **before** we create real datetimes. If dates were converted first, `clean_text` would turn them back into strings.

For the timestamps: `utc=True` reads the offset, `tz_convert("Asia/Kolkata")` shifts to India time, and `tz_localize(None)` removes the timezone label so they match legacy.

**What we used:** `clean_text`, `pd.to_datetime(utc=True)`, `.dt.tz_convert`, `.dt.tz_localize(None)`.

In [64]:
# text cleanup first, THEN dates (cleaning turns everything into strings)
nova = clean_text(nova)

for c in ["created_at", "slot_at"]:
    nova[c] = (pd.to_datetime(nova[c], errors="coerce", utc=True)
                 .dt.tz_convert("Asia/Kolkata")
                 .dt.tz_localize(None))

nova["source_system"] = "nova"
nova.head()

,test_code,test_price,booking_id,created_at,slot_at,city,clinic_id,channel,source,home_fee,rescheduled_from,status_at_export,source_file,patient_id,patient.is_new,source_system
0,T007,550.0,nova_che_0087556,2026-04-21 01:57:00,2026-04-25 10:16:00,Chennai,CHE-01,CLINIC,WALKIN,0,<NA>,booked,datapack/bookings/nova/chennai/bookings_2026-0...,PCHE0044922,False,nova
1,T007,550.0,nova_che_0087557,2026-03-28 03:02:00,2026-04-01 09:17:00,Chennai,CHE-03,CLINIC,PHONE,0,<NA>,booked,datapack/bookings/nova/chennai/bookings_2026-0...,PCHE0047035,False,nova
2,T011,750.0,nova_che_0087557,2026-03-28 03:02:00,2026-04-01 09:17:00,Chennai,CHE-03,CLINIC,PHONE,0,<NA>,booked,datapack/bookings/nova/chennai/bookings_2026-0...,PCHE0047035,False,nova
3,T004,650.0,nova_che_0087557,2026-03-28 03:02:00,2026-04-01 09:17:00,Chennai,CHE-03,CLINIC,PHONE,0,<NA>,booked,datapack/bookings/nova/chennai/bookings_2026-0...,PCHE0047035,False,nova
4,T004,650.0,nova_che_0087558,2026-04-28 04:46:00,2026-04-30 08:19:00,Chennai,CHE-05,CLINIC,PHONE,0,<NA>,booked,datapack/bookings/nova/chennai/bookings_2026-0...,PCHE0051652,False,nova


## 6. Stack legacy + nova into booking lines

Now both systems share column names, so we can stack them.

### Stack, standardise, convert money, free memory

- `pd.concat([legacy, nova])` stacks the two tables
- lower-case `channel` and `source` so `Home` and `home` are one value
- normalise the city, convert prices and fees to numbers
- `free(...)` releases the three source frames (we now only need `lines`)
- `value_counts()` on city is a quick check that the names look right

**What we used:** `pd.concat(ignore_index=True)`, `.str.lower()`, our helper functions `norm_city`, `to_num`, `free`, `value_counts()`.

In [65]:
lines = pd.concat([legacy, nova], ignore_index=True)

for c in ["channel", "source"]:
    lines[c] = lines[c].str.lower()
lines["city"] = norm_city(lines["city"])
lines["test_price"] = to_num(lines["test_price"])
lines["home_fee"]   = to_num(lines["home_fee"])
free("legacy", "nova", "rows", "p")   # lines now holds everything; release the three source frames
lines["city"].value_counts()

,count
city,
Bengaluru,301219
Chennai,243546
Ahmedabad,235373
Hyderabad,220989
Pune,185149
Jaipur,128928


### Remove duplicates and flag missing tests

1. **Exact duplicate rows** are dropped
2. **Same (booking, test) twice but different elsewhere** = a *conflict*. We only **count** these, to look at them later
3. Lines with **no `test_code`** are kept (they still carry a price) but **flagged** in `test_code_missing`

> Note: `source_file` is still a column here, so two identical rows from **different files** are *not* treated as duplicates. Drop it before `drop_duplicates()` if you want those removed too.

**What we used:** `drop_duplicates()`, `duplicated([cols], keep=False)` (marks **all** members of a duplicate group), `&`, `.isna()`.

In [66]:
before = len(lines)
lines = lines.drop_duplicates()
print(f"dropped {before - len(lines):,} exact duplicate rows")

# same (booking, test) twice but with different values = a conflict worth looking at
dup = lines.duplicated(["booking_id", "test_code"], keep=False) & lines["test_code"].notna()
print(f"{dup.sum()} rows share (booking_id, test_code) but differ elsewhere")

lines["test_code_missing"] = lines["test_code"].isna()
print(f"{lines['test_code_missing'].sum():,} lines have no test_code (kept, flagged)")

dropped 0 exact duplicate rows
0 rows share (booking_id, test_code) but differ elsewhere
1,311 lines have no test_code (kept, flagged)


### Attach the catalogue and compare prices

`join_report` adds the catalogue's name and list price to every line (and prints how many lines had no catalogue match). `price_vs_list_rs` = price charged minus list price: a non-zero value means a discount or surcharge was recorded **on the line**.

**What we used:** `join_report`, column subtraction, selecting columns with a list `df[[...]]`.

In [67]:
# attach the catalogue, then compare price charged vs list price
lines = join_report(lines, cat, "test_code", "lines->catalogue")
lines["price_vs_list_rs"] = lines["test_price"] - lines["list_price_rs"]
lines[["booking_id", "test_code", "test_price", "list_price_rs", "price_vs_list_rs"]].head()

[lines->catalogue] 1,311 of 1,315,204 rows unmatched on test_code


,booking_id,test_code,test_price,list_price_rs,price_vs_list_rs
0,BK-AHM-202501-000001,T009,800.0,800,0.0
1,BK-AHM-202501-000001,T004,650.0,650,0.0
2,BK-AHM-202501-000001,T007,550.0,550,0.0
3,BK-AHM-202501-000002,T004,650.0,650,0.0
4,BK-AHM-202501-000003,T003,450.0,450,0.0


## 7. `results/`: one row per test the lab ran

The key is `(booking_id, test_code)`. Steps:

1. read all city/month files
2. keep the **first** of any duplicate key
3. save rows whose `booking_id` is not in the bookings (**orphans**)
4. left-join the rest onto the lines

### Read, dedupe, find orphans

Why a `set`? `x.isin(set(...))` is much faster than `isin(list)` or `isin(series)` on large tables.

`~dup` means "not a duplicate", so `res[~dup]` keeps the first copy.

**What we used:** `read_many`, `duplicated(keep="first")`, boolean indexing with `~`, `set()`, `.isin`, `.to_csv(index=False)`.

In [68]:
res = read_many(f"{ROOT}/results/*/*.csv.gz")

dup = res.duplicated(["booking_id", "test_code"], keep="first")
print(f"{dup.sum()} duplicate (booking_id, test_code) rows: keeping first")
res = res[~dup]

orph = ~res["booking_id"].isin(set(lines["booking_id"]))
print(f"{orph.sum():,} orphan result rows")
res[orph].to_csv(f"{OUT}/orphans_results.csv.gz", index=False)

/content/datapack/results/*/*.csv.gz: 108 files, 1,078,420 rows
9 duplicate (booking_id, test_code) rows: keeping first
5,401 orphan result rows


### Join results onto the lines

We leave out `source_file` so it does not create a clashing column.

**What we used:** list comprehension with `if`, `merge(on=[two keys], how="left")`, `.shape`.

In [69]:
res_cols = [c for c in res.columns if c != "source_file"]
lines = lines.merge(res[res_cols], on=["booking_id", "test_code"], how="left")
lines.shape

(1315204, 25)

## 8. Booking-level table (one row per booking)

`lines` has **one row per booking x test**. Many analyses need **one row per booking**, so we:

1. drop the line-level columns and keep the first row per `booking_id`
2. use `groupby` to count tests and sum prices per booking
3. join patients and clinics

`count()` ignores blanks, so bookings with no test get `n_tests = 0`.

### Collapse lines to bookings

**What we used:** list of column names to exclude, list comprehension with `not in`, `drop_duplicates("booking_id")`, `groupby().agg(name=(column, function))`, `reset_index()`, `merge`.

In [70]:
line_only = ["test_code", "test_price", "test_name", "list_price_rs", "is_package",
             "price_vs_list_rs", "test_code_missing", "result_ts", "result_value",
             "result_flag", "result_value_raw", "result_unit", "source_file"]
booking_cols = [c for c in lines.columns if c not in line_only]

bk = lines[booking_cols].drop_duplicates("booking_id").copy()

agg = (lines.groupby("booking_id")
            .agg(n_tests=("test_code", "count"), tests_total=("test_price", "sum"))
            .reset_index())
bk = bk.merge(agg, on="booking_id", how="left")
print(bk.shape, "| unique bookings:", bk["booking_id"].nunique())

(568007, 16) | unique bookings: 568007


### Join patients and clinics

- `join_report` shows how many bookings had no patient or no clinic
- the clinic columns get a `clinic_` prefix (except the key `clinic_id`), and clinic `city` is dropped, so nothing clashes with the **booking's** city
- `ids` is a `set` of every booking id. **Every orphan check below reuses it**

**What we used:** dictionary comprehension for renaming, `drop(columns=...)`, `set(...)`.

In [71]:
bk = join_report(bk, patients_slim, "patient_id", "bookings->patients")

# prefix clinic columns (except the key) and drop clinic city to avoid clashing with booking city
clinics_slim = (clinics.drop(columns="city")
                       .rename(columns={c: f"clinic_{c}" for c in clinics.columns
                                        if c not in ("clinic_id", "city")}))
bk = join_report(bk, clinics_slim, "clinic_id", "bookings->clinics")

ids = set(bk["booking_id"])   # reused by every orphan check below
bk.head()

[bookings->patients] 0 of 568,007 rows unmatched on patient_id
[bookings->clinics] 0 of 568,007 rows unmatched on clinic_id


,booking_id,created_at,slot_at,city,clinic_id,channel,source,patient_id,patient_name,home_fee,...,patient.is_new,n_tests,tests_total,patient_phone,patient_dob,patient_signup_date,clinic_clinic_name,clinic_opened_on,clinic_has_lab,clinic_slots_per_day
0,BK-AHM-202501-000001,2025-01-08 03:16:00,2025-01-09 08:26:00,Ahmedabad,AHM-03,clinic,walkin,PAHM0028525,Saanvi Bose,0,...,<NA>,3,2000.0,+91 76866 11670,1995-06-17,2024-01-11,Kalpa Health Ahmedabad 3,2023-06-01,0,40
1,BK-AHM-202501-000002,2025-01-03 05:18:00,2025-01-06 13:22:00,Ahmedabad,AHM-04,clinic,app,PAHM0031384,Nisha Iyer,0,...,<NA>,1,650.0,+91 75793 12879,1997-03-17,2024-06-22,Kalpa Health Ahmedabad 4,2023-06-01,0,40
2,BK-AHM-202501-000003,2025-01-27 10:46:00,2025-01-28 13:37:00,Ahmedabad,AHM-01,home,app,PAHM0027001,Sneha Menon,150,...,<NA>,4,1670.0,+91 94881 71237,1977-10-06,2023-10-23,Kalpa Health Ahmedabad 1,2023-06-01,1,60
3,BK-AHM-202501-000004,2025-01-20 02:48:00,2025-01-24 07:15:00,Ahmedabad,AHM-03,clinic,app,PAHM0007069,Arjun Bose,0,...,<NA>,1,160.0,+91 96126 68447,1965-06-14,2024-11-12,Kalpa Health Ahmedabad 3,2023-06-01,0,40
4,BK-AHM-202501-000005,2025-01-06 08:21:00,2025-01-11 09:56:00,Ahmedabad,AHM-03,clinic,app,PAHM0002520,Ritika Nair,0,...,<NA>,2,1900.0,+91 83270 41852,1986-10-07,2024-12-31,Kalpa Health Ahmedabad 3,2023-06-01,0,40


## 9. `collections/`: sample collection events

Same recipe: stack, drop exact duplicates, parse the timestamp, save orphans. Then bring in `patient_id`, `city` and `slot_at` from the booking.

### Read, dedupe, parse, find orphans

We drop `source_file` **before** `drop_duplicates()`, so identical rows from different files are removed too.

**What we used:** `read_many`, `len()` before and after to count what was removed, `drop_duplicates()`, `pd.to_datetime(errors="coerce")`, `isin(ids)`, `.to_csv`.

In [72]:
col = read_many(f"{ROOT}/collections/*.csv.gz")
n = len(col)
col = col.drop(columns="source_file").drop_duplicates()
print(f"dropped {n - len(col):,} duplicate rows")

col["collected_at"] = pd.to_datetime(col["collected_at"], errors="coerce")

orph = ~col["booking_id"].isin(ids)
print(f"{orph.sum():,} orphan collection rows")
col[orph].to_csv(f"{OUT}/orphans_collections.csv.gz", index=False)

/content/datapack/collections/*.csv.gz: 108 files, 552,492 rows
dropped 2,894 duplicate rows
0 orphan collection rows


### Join booking details and look at the statuses

`value_counts(dropna=False)` shows blanks too, which `value_counts()` would hide. The status is `collected` or `no_show`.

**What we used:** `merge(how="left")` with a column list, `value_counts(dropna=False)`.

In [73]:
col = col.merge(bk[["booking_id", "patient_id", "city", "slot_at"]], on="booking_id", how="left")
col["status"].value_counts(dropna=False)

,count
status,
collected,470426
no_show,79172


## 10. `invoices/`

Amount to a number, date to datetime, orphan check. We do **not drop** orphans from the invoice table. We only copy them to a separate file, so revenue totals stay complete and the problem rows are still visible.

**What we used:** `to_num`, `pd.to_datetime(errors="coerce")`, `~isin(ids)`, f-string with `{:,}`, `.to_csv`.

In [74]:
inv = read_many(f"{ROOT}/invoices/*.csv.gz")

inv["amount"] = to_num(inv["amount"])
inv["invoice_date"] = pd.to_datetime(inv["invoice_date"], errors="coerce")

orph = ~inv["booking_id"].isin(ids)
print(f"{orph.sum():,} of {len(inv):,} invoices point at no booking")
inv[orph].to_csv(f"{OUT}/orphans_invoices.csv.gz", index=False)
inv.head()

/content/datapack/invoices/*.csv.gz: 7 files, 470,776 rows
0 of 470,776 invoices point at no booking


,invoice_id,invoice_date,booking_id,patient_id,city,channel,amount,source_file
0,INV-2024-000001,2024-12-30,BK-BLR-202501-006763,PBLR0044005,Bengaluru,clinic,280.0,datapack/invoices/invoices_2024Q4.csv.gz
1,INV-2024-000002,2024-12-30,BK-BLR-202501-007103,PBLR0044345,Bengaluru,clinic,1070.0,datapack/invoices/invoices_2024Q4.csv.gz
2,INV-2024-000003,2024-12-31,BK-PUN-202501-001985,PPUN0008860,Pune,clinic,350.0,datapack/invoices/invoices_2024Q4.csv.gz
3,INV-2024-000004,2024-12-31,BK-CHE-202501-001176,PCHE0012203,Chennai,home,3299.0,datapack/invoices/invoices_2024Q4.csv.gz
4,INV-2025-000001,2025-01-01,BK-AHM-202501-005119,PAHM0034226,Ahmedabad,clinic,2649.0,datapack/invoices/invoices_2025Q1.csv.gz


## 11. `payments/`: the messiest folder

Two kinds of file:

- **gateway exports**: one row per payment attempt. Amounts are in **paise** (divide by 100) and time is **epoch seconds** (seconds since 1 Jan 1970)
- **settlements**: what the bank paid out, joined by `settlement_id`

The gateway's invoice reference is typed inconsistently (`INV-2025-000001`, `inv-2025-000001`, `INV2025000001`), so we **normalise** it before joining to invoices.

### How the invoice-id normaliser works (on a toy series)

1. upper-case and strip everything that is not a letter or digit
2. check it is exactly `INV` + 4 digits + 6 digits
3. if valid, rebuild the standard form `INV-YYYY-NNNNNN`; if not, **leave the original untouched**

The fourth toy value has 7 digits at the end, so it is **not valid** and stays as typed. That is on purpose: we do not guess.

**What we used:** `.str.upper()`, `.str.replace(r"[^A-Z0-9]", "", regex=True)` (regular expression: "anything except letters and digits"), `.str.fullmatch(r"INV\d{4}\d{6}")`, `Series.where(condition, other)`, string slicing with `.str[3:7]`.

In [75]:
# how the invoice-id normaliser works, on a toy series
s = pd.Series(["INV-2025-000001", "inv-2025-000001", "INV2025000001", "INV-2025-0000012"])

k = s.str.upper().str.replace(r"[^A-Z0-9]", "", regex=True)   # strip dashes/spaces -> INV2025000001
ok = k.str.fullmatch(r"INV\d{4}\d{6}")                        # exactly INV + 4 digits + 6 digits?
fixed = s.where(~ok, "INV-" + k.str[3:7] + "-" + k.str[7:])   # rebuild INV-YYYY-NNNNNN where valid
pd.DataFrame({"raw": s, "valid": ok, "fixed": fixed})

,raw,valid,fixed
0,INV-2025-000001,True,INV-2025-000001
1,inv-2025-000001,True,INV-2025-000001
2,INV2025000001,True,INV-2025-000001
3,INV-2025-0000012,False,INV-2025-0000012


### Gateway export: paise to rupees, epoch to datetime, clean invoice id

- `amount_rs` = paise / 100
- `created_at`: epoch seconds to a UTC datetime, then to India time, then drop the timezone
- keep the original id as `invoice_id_raw` **for auditing**, and normalise the working one
- rename `notes_invoice_ref` to `invoice_id` and `status` to `payment_status`, and drop the raw paise and epoch columns

**What we used:** `norm_invoice_id` (a function), `to_num`, `pd.to_datetime(unit="s", utc=True)`, `.dt.tz_convert`, `.dt.tz_localize(None)`, `.rename`, `.drop`.

In [76]:
def norm_invoice_id(s):
    k = s.str.upper().str.replace(r"[^A-Z0-9]", "", regex=True)
    ok = k.str.fullmatch(r"INV\d{4}\d{6}")
    return s.where(~ok, "INV-" + k.str[3:7] + "-" + k.str[7:])

In [77]:
gw = read_many(f"{ROOT}/payments/gateway_export_*.csv.gz")

gw["amount_rs"] = to_num(gw["amount_paise"]) / 100
gw["created_at"] = (pd.to_datetime(to_num(gw["created_at_epoch"]), unit="s", utc=True)
                      .dt.tz_convert("Asia/Kolkata").dt.tz_localize(None))

gw["invoice_id_raw"] = gw["notes_invoice_ref"]            # keep the original for auditing
gw["notes_invoice_ref"] = norm_invoice_id(gw["notes_invoice_ref"])

gw = (gw.rename(columns={"notes_invoice_ref": "invoice_id", "status": "payment_status"})
        .drop(columns=["amount_paise", "created_at_epoch"]))
gw[["invoice_id_raw", "invoice_id", "amount_rs", "created_at", "payment_status"]].head()

/content/datapack/payments/gateway_export_*.csv.gz: 20 files, 473,507 rows


,invoice_id_raw,invoice_id,amount_rs,created_at,payment_status
0,INV-2024-000002,INV-2024-000002,1070.0,2024-12-30 22:19:38,captured
1,INV-2024-000003,INV-2024-000003,350.0,2024-12-31 19:35:47,captured
2,INV-2024-000001,INV-2024-000001,280.0,2025-01-04 18:44:28,captured
3,INV-2024-000004,INV-2024-000004,3299.0,2025-01-05 23:19:48,captured
4,inv-2025-000001,INV-2025-000001,2649.0,2025-01-15 20:23:25,captured


### Settlements: paise to rupees

- loop over `amount`, `fees`, `tax`: each `<name>_paise` column becomes `<name>_rs`
- rename the settlement `amount_rs` to `settlement_amount_rs` so it does not clash with the gateway's `amount_rs`
- `settled_at` from epoch seconds, then drop the raw columns
- join onto the gateway rows by `settlement_id`

**What we used:** a loop that builds column names with f-strings, `to_num`, `.rename`, `.drop`, `join_report`.

In [78]:
st = read_many(f"{ROOT}/payments/settlements_*.csv.gz")

for c in ["amount", "fees", "tax"]:
    st[f"{c}_rs"] = to_num(st[f"{c}_paise"]) / 100        # amount_rs, fees_rs, tax_rs
st = st.rename(columns={"amount_rs": "settlement_amount_rs"})
st["settled_at"] = (pd.to_datetime(to_num(st["created_at_epoch"]), unit="s", utc=True)
                      .dt.tz_convert("Asia/Kolkata").dt.tz_localize(None))
st = (st.rename(columns={"status": "settlement_status"})
        .drop(columns=["amount_paise", "fees_paise", "tax_paise", "created_at_epoch", "source_file"]))

pay = join_report(gw, st, "settlement_id", "gateway->settlements")

/content/datapack/payments/settlements_*.csv.gz: 8 files, 562 rows
[gateway->settlements] 18,038 of 473,507 rows unmatched on settlement_id


### Attach invoices, then look for double charges

The payments table gets the invoice's `booking_id` and `invoice_amount_rs`. `drop_duplicates("invoice_id")` first guarantees one row per invoice, so the join cannot multiply payments.

Finally: for **captured** payments only, count per invoice. An invoice captured more than once is either a **duplicate charge or a part-payment**. This count tells you how many invoices to investigate; it does not tell you which kind each one is.

**What we used:** `join_report`, `drop_duplicates`, `groupby().size()`, comparison on a Series `cap > 1`.

In [79]:
inv_small = (inv[["invoice_id", "booking_id", "amount"]]
             .drop_duplicates("invoice_id")
             .rename(columns={"amount": "invoice_amount_rs"}))
pay = join_report(pay, inv_small, "invoice_id", "payments->invoices")

# an invoice captured more than once = duplicate charge or part-payments
cap = pay[pay["payment_status"] == "captured"].groupby("invoice_id").size()
print(f"{(cap > 1).sum():,} invoices have more than one captured payment")

[payments->invoices] 4,765 of 473,507 rows unmatched on invoice_id
1,925 invoices have more than one captured payment


## 12. `campaigns/`: tags joined to the campaign table

`campaign_bookings` says **which booking got which campaign**, and `campaigns` holds each campaign's details (including a default waiver). We join them on `campaign_id`. We rename the campaign-level `waiver_rs` to `campaign_default_waiver_rs` so it is never confused with a per-booking waiver column.

**What we used:** `read_csv_any`, `merge(on=..., how="left")`, `.rename(columns=...)` inside a merge.

In [80]:
camp = read_csv_any(f"{ROOT}/campaigns/campaigns.csv")
cb   = read_csv_any(f"{ROOT}/campaigns/campaign_bookings.csv.gz")

# rename the campaign-level waiver so it is not confused with a per-booking waiver column
cb = cb.merge(camp.rename(columns={"waiver_rs": "campaign_default_waiver_rs"}),
              on="campaign_id", how="left")
print(f"{len(cb):,} campaign rows")
cb.head()

48,869 campaign rows


,booking_id,campaign_id,waiver_rs,campaign_name,starts,ends,campaign_default_waiver_rs
0,BK-AHM-202601-065286,HC-FREE-2026Q1,150,"Free home collection, Jan to Mar 2026",2026-01-01,2026-03-31,150
1,BK-AHM-202601-065292,HC-FREE-2026Q1,150,"Free home collection, Jan to Mar 2026",2026-01-01,2026-03-31,150
2,BK-AHM-202601-065298,HC-FREE-2026Q1,150,"Free home collection, Jan to Mar 2026",2026-01-01,2026-03-31,150
3,BK-AHM-202601-065301,HC-FREE-2026Q1,150,"Free home collection, Jan to Mar 2026",2026-01-01,2026-03-31,150
4,BK-AHM-202601-065306,HC-FREE-2026Q1,150,"Free home collection, Jan to Mar 2026",2026-01-01,2026-03-31,150


## 13. `plan/`: wide to long

The plan has **one column per month** (`2026-01`, `2026-02`, ...). That is easy for a person to read but awkward for grouping and plotting. `melt` turns the month columns into rows: `city, metric, month, target`.

- `id_vars` stay as they are
- `var_name` is the new column that holds the old column names
- `value_name` is the new column that holds the values

**What we used:** `read_many(add_source=False)`, `DataFrame.melt(id_vars, var_name, value_name)`, `norm_city`, `to_num`.

In [81]:
plan = read_many(f"{ROOT}/plan/*.csv", add_source=False)

plan_long = plan.melt(id_vars=["city", "metric"], var_name="month", value_name="target")
plan_long["city"] = norm_city(plan_long["city"])
plan_long["target"] = to_num(plan_long["target"])
plan_long.head()

/content/datapack/plan/*.csv: 1 files, 6 rows


,city,metric,month,target
0,Ahmedabad,test_volume_target_2026,2026-01,15119
1,Bengaluru,test_volume_target_2026,2026-01,20380
2,Chennai,test_volume_target_2026,2026-01,15474
3,Hyderabad,test_volume_target_2026,2026-01,14840
4,Jaipur,test_volume_target_2026,2026-01,8853


## 14. Write the outputs

Every clean table is saved as a gzipped CSV on **Drive** (`OUT`). Pandas compresses automatically because the filename ends in `.csv.gz`. `index=False` leaves out the row numbers, and `source_file` is dropped from `lines` because it was only for tracing.

**What we used:** `.to_csv(path, index=False)`, `errors="ignore"` in `drop`, `sorted(os.listdir(OUT))` to confirm the files exist.

In [82]:
lines.drop(columns="source_file", errors="ignore").to_csv(f"{OUT}/booking_lines.csv.gz", index=False)
bk.to_csv(f"{OUT}/bookings.csv.gz", index=False)
col.to_csv(f"{OUT}/collections.csv.gz", index=False)
inv.to_csv(f"{OUT}/invoices.csv.gz", index=False)
pay.to_csv(f"{OUT}/payments.csv.gz", index=False)
cb.to_csv(f"{OUT}/campaigns.csv.gz", index=False)
plan_long.to_csv(f"{OUT}/plan_long.csv.gz", index=False)
print(sorted(os.listdir(OUT)))

['booking_lines.csv.gz', 'bookings.csv.gz', 'campaigns.csv.gz', 'collections.csv.gz', 'invoices.csv.gz', 'orphans_collections.csv.gz', 'orphans_invoices.csv.gz', 'orphans_results.csv.gz', 'payments.csv.gz', 'plan_long.csv.gz']


## Things to check in the original script

*(These notes come from your original notebook.)*

- **Cleaning order for nova.** The original runs `clean_text` after converting the dates, which would turn those timestamps back into strings. Here the order is swapped. Check the original's output types if the two differ
- **Invoice ids with a 7-digit suffix** do not match the normaliser and will show up as unmatched in `payments->invoices`. The script comments say they could not be reconciled by amount
- **`cities.csv`** is loaded but never used
- **`verify()`** only loads `MANIFEST.json`. It does not check hashes
- **`clean_text` header-echo check** also drops fully blank rows in the original (NA comparisons count as "all equal"). Here `.fillna(False)` keeps them

**Two more things I noticed while reading the code** (from reading only; the data was not available to test):

- **Nova columns come from the first record only.** `meta_cols` is built from `rows[0]`. `errors="ignore"` protects against a *missing* key but not against a *new* key that first appears in a later record, which would be dropped silently
- **Booking-level columns come from the first line of each booking.** `drop_duplicates("booking_id")` keeps one line's booking fields. If two lines of one booking disagree on a booking-level field, the others are lost without a message

## 15. Sanity checks on the merged data

The original notebook describes this part as answering the open questions from the merge run: which cities come from which system, what the unmatched payment references look like, and whether repeated captures are part-payments or duplicates. **Only the first of those checks (cities by system) is present in the notebook you sent.** The other two would be new cells (for example, list the invoice references that stayed unmatched, and compare the amounts of repeated captures to the invoice amount).

### Cities by source system

Legacy should cover all six cities and nova only some. If a city appears in **both** systems, there was a cutover during the period, which matters when you compare volumes over time.

**What we used:** `groupby([two columns]).size()`, `value_counts()`.

In [83]:
# cities by source system: legacy should cover all six, nova only some
print(lines.groupby(["source_system", "city"]).size(), "\n")
print("bookings by city:\n", bk["city"].value_counts())

source_system  city     
legacy         Ahmedabad    235373
               Bengaluru    301219
               Chennai      202117
               Hyderabad    220989
               Jaipur       107918
               Pune         185149
nova           Chennai       41429
               Jaipur        21010
dtype: int64 

bookings by city:
 city
Bengaluru    130181
Chennai      105176
Ahmedabad    101583
Hyderabad     95335
Pune          79979
Jaipur        55753
Name: count, dtype: int64


## Recap: what we used

| Tool | Where |
|---|---|
| `dtype=str` + `clean_text` | read everything as text, convert on purpose |
| `try / except`, `for ... in (...)` | encoding fallback |
| `glob`, `pd.concat` | read and stack many files |
| `pd.to_datetime(format=..., errors="coerce")` | legacy dates |
| `json.loads`, `pd.json_normalize` | nested nova bookings |
| `tz_convert`, `tz_localize(None)` | timezone to India time |
| `drop_duplicates`, `duplicated(keep=...)` | removing repeats |
| `merge(indicator=True)` in `join_report` | join with a "how many unmatched" report |
| `isin(set(...))` + `.to_csv` | orphan checks that save the bad rows |
| `groupby().agg(...)` | lines to bookings |
| regex with `.str.replace`, `.str.fullmatch` | invoice-id normaliser |
| `Series.where` | change only the rows that pass a test |
| `melt` | wide plan to long plan |
| `assert` | fail early with a clear message |

**Honest caveats**
- This is the original logic re-cut into small cells with explanations. It has not been run here because the data pack was not available. It is built for Google Colab (it mounts Drive).
- The "Things to check" notes about the *original script* are from your notebook. I could not compare against the original script itself.